Ch 4-6 Trees, Graphs, and Algebra (Probability)

* [Ch 1-3: Array, Strings, Linked Lists, Stacks, and Queues](https://colab.research.google.com/github/henry4j/-/blob/main/episode_1_3.ipynb)
* https://github.com/henry4j/_/blob/master/man/episode4-7.ipynb

In [ ]:
# @title ##### 4.1 Given a directed graph, design an algorithm to find whether there is a route between two nodes.
from collections import namedtuple
from enum import IntFlag
EType = IntFlag("EType", "ENTER CROSS LEAVE")

class Edge(namedtuple("Edge", ["to", "weight"], defaults=[0])):
  def __repr__(self):
    return f"Edge({self.to!r}{(w := self.weight) and f', {w!r}' or ''})"

def DFS(graph, vertex, entered=None):
  entered = entered or set()
  if (from_ := vertex) not in entered and not entered.add(vertex):
    # yield EType.ENTER, (edge := Edge(v)), None
    for e in graph[from_] or []:
      yield EType.CROSS, e, from_
      yield from DFS(graph, e.to, entered)
    # yield EType.LEAVE, edge, None

# graph:
# B1 ← C2 → A0
# ↓  ↗
# D3 ← E4
graph = [[]] * 5
graph[0] = []  # out-degree of 0
graph[1] = [Edge(3)]  # B1 → D3
graph[2] = [Edge(0), Edge(1)]  # C2 → A0, C2 → B1
graph[3] = [Edge(2)]  # D3 → C2
graph[4] = [Edge(3)]  # E4 → D3

def can_reach(source, sink, graph) -> bool:
  return any(edge.to == sink for type_, edge, from_ in DFS(graph, source))

assert all(can_reach(4, sink, graph) for sink in range(3))
assert not any(can_reach(source, 4, graph) for source in (0, 3))

In [ ]:
# @title ##### 4.2 Given a sorted (increasing order) array, design an algorithm to create a binary search tree with the minimal height.
from collections import namedtuple
from dataclasses import dataclass
from itertools import dropwhile

class BTree:
  def __init__(self, value=None, left=None, right=None, parent=None):
    self.value, self.left, self.right, self.parent = value, left, right, parent
  def __repr__(self):
    n = len(values := list(vars(self).values()))
    i = next((i for i, e in enumerate(reversed(values)) if e is not None), n)
    return f"BTree({repr(values[:n-i])[1:-1]})"
  @classmethod
  def from_values(cls, values, start=0, stop=None):
    if stop is None:
      stop = len(values)
    if stop - start > 0:
      mid = (start + stop - 1) // 2
      l = BTree.from_values(values, start, mid)
      r = BTree.from_values(values, mid + 1, stop)
      return cls(values[mid], l, r, None)
  def set_parent(self):  # set the parent fields on both the left and right children.
    for e in (self.left, self.right):
      if e is not None:
        e.parent = self
        e.set_parent()

# tree:  4
#   / \
#  2   6
#   1 3 5 7
t7 = BTree.from_values((1, 2, 3, 4, 5, 6, 7))
assert "BTree(4, BTree(2, BTree(1), BTree(3)), BTree(6, BTree(5), BTree(7)))" == repr(t7)  # fmt: skip

In [ ]:
# @title ##### 4.3 Given a binary tree, design an algorithm to creates a linked list of all the nodes at each depth, e.g., if you have a tree with depth D, you will have D linked lists.
def to_d_linked_lists(tree_of_depth_d):
  LL = []  # output: a list of lists.
  q = [tree_of_depth_d]
  while q:
    p = []
    for i, e in enumerate(q):
      for c in (e.left, e.right):
        if c is not None:
          p.append(c)
      e.left = None if i == 0 else q[i - 1]
      e.right = q[i + 1] if i < len(q) else None
    LL.append(q[0])
    q = p
  return LL

In [ ]:
# @title ##### 4.4 Implement a function to check if a binary tree is balanced. For the purposes of this question, a balanced tree is defined to be a tree such that the heights of two subtrees of any node never differ by more than one.
# tree: a
# . .  /   \
# .   b  .  f
#   c  e   g
#  d
c = BTree("c", BTree("d"))
e = BTree("e")
b = BTree("b", c, e)
a = BTree("a", b, BTree("f"))

def is_balanced(tree):  # returns (balanced, height)
  if tree is None:
    return (True, 0)
  L, R = is_balanced2(tree.left), is_balanced2(tree.right)
  b = L[0] and R[0] and abs(L[1] - R[1]) < 2
  h = 1 + max(L[1], R[1])
  return (b, h)

assert not is_balanced(a)[0]
a.right.left = BTree("g")
assert is_balanced(a)[0]

In [ ]:
# @title ##### 4.5 Implement a function to check if a binary tree is a binary search tree.
# def is_ordered(tree):
#   def process(v):
#   process.pred, pred = v, process.pred
#   process.ordered &= pred is None or pred.value <= v.value
#   def may_enter(v):
#   return process.ordered
#   process.ordered, process.pred = (
#   True,
#   None,
#   )  # ordered is assumed to be True; predecessor is None.
#   tree.order(process, may_enter)
#   return process.ordered

def ordered(tree):  # returns (ordered, min, max)
  if tree is None:
    return (True, None, None)
  L, R = ordered(tree.left), ordered(tree.right)
  O = (
    L[0]
    and R[0]
    and (L[2] is None or L[2] <= tree.value)
    and (R[1] is None or R[1] >= tree.value)
  )
  min_ = tree.value if L[1] is None else L[1]
  max_ = tree.value if R[2] is None else R[2]
  return (O, min_, max_)

def ordered2(tree, min=None, max=None):
  return (
    tree is None
    or (min is None or tree.value >= min)
    and (max is None or tree.value <= max)
    and ordered2(tree.left, min, tree.value)
    and ordered2(tree.right, tree.value, max)
  )

# assert is_ordered(BTree.from_values((1, 2, 3, 4, 5, 6, 7)))
assert ordered(BTree.from_values((1, 2, 3, 4, 5, 6, 7)))[0]
assert not ordered(BTree.from_values((1, 2, 3, 4, 0, 6, 7)))[0]
assert ordered2(BTree.from_values((1, 2, 3, 4, 5, 6, 7)))

In [ ]:
from collections import deque
from itertools import dropwhile

class BTree:
  def order(self, process, may_enter=None, leave=None):  # for traversals
    if not may_enter or may_enter(self):
      self.left and self.left.order(process, may_enter, leave)
      process and process(self)
      self.right and self.right.order(process, may_enter, leave)
      leave and leave(self)
  def order2(self, process):
    v, stack = self, []
    while v or stack:
      if v:
        stack.append(v)
        v = v.left
      else:
        v = stack.pop()
        process(v)
        v = v.right
  def dfs(self, may_enter=None, leave=None):  # for traversals
    if not may_enter or may_enter(self):
      for w in (self.left, self.right):
        w and w.dfs(may_enter, leave)
      leave and leave(self)
  def bfs(self, may_enter=None, leave=None):
    may_enter = may_enter or (lambda *_, **__: True)
    q = deque()
    q.append(self)  # enque, or offer
    while q:
      v = q.popleft()  # deque, or poll
      if may_enter(v):
        for w in (v.left, v.right):
          w and q.append(w)
      leave and leave(v)